# Infosys Quarterly Report Analysis using RAG

## Section 1: Installation and Setup

We will use the following libraries. Please install it in your virtual environment using `pip install <package_name>`

- **getpass4**: A secure utility used to prompt users for passwords or API tokens in the terminal without echoing their keystrokes.
- **pypdf**: A pure-Python library designed for splitting, merging, cropping, and extracting text or metadata from PDF files.
- **faiss-cpu**: Facebook AI Similarity Search (FAISS), a highly efficient library for dense vector similarity searching and clustering optimized for CPUs.
- **llama-index**: A data orchestration framework designed to connect custom data sources to Large Language Models (LLMs) for building RAG applications.
- **llama-index-readers-file**: A plugin for LlamaIndex that provides native data loaders to read and parse local file formats like PDFs, DOCX, and TXT files.
- **llama-index-vector-stores-faiss**: A LlamaIndex integration bridge that allows developers to use a FAISS index as the underlying vector storage engine.
- **llama-index-llms-groq**: A LlamaIndex integration that connects your pipeline to Groq's high-speed, hardware-accelerated LLM inference engine. (If you want to use OpenAI then install `llama-index-llms-openai`)
- **llama-index-embeddings-huggingface**: A LlamaIndex extension that lets you generate text embeddings locally using open-source models hosted on Hugging Face.
- **docling**: A document conversion tool that parses complex document layouts, preserving tables, headers, and structures with high fidelity.
- **ragas**: An evaluation framework specifically built to assess and score Retrieval-Augmented Generation (RAG) pipelines using LLM-as-a-judge metrics.

### What is LlamaIndex?

**LlamaIndex** is a data framework designed specifically for building LLM-based applications. While LLMs are trained on massive public datasets, they lack access to your private, custom data (like the uploaded ifrs-inr-press-release.pdf). LlamaIndex bridges this gap by acting as the data ingest, management, and retrieval orchestrator.

**Core Capabilities:**
- Data Ingestion (Loaders): Seamlessly connects to various data sources (PDFs, APIs, databases) and converts them into standardized Node objects.

- Data Indexing: Structures data into searchable vectors, keyword tables, or hierarchical graphs.

- Query Interface: Provides advanced retrieval mechanisms, enabling developers to feed relevant contexts into an LLM dynamically.

- Agentic Framework: Supports building multi-step reasoning agents that interact with external data environments.

## Section-2: Configure LLM Service and Embedding Model

In [1]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [2]:
from llama_index.llms.groq import Groq
llm = Groq(model="llama-3.3-70b-versatile", temperature=0.1, max_tokens=512)

# If you want to use OpenAI, you can set it up like this:
# from llama_index.llms.openai import OpenAI   
# llm = OpenAI(model="gpt-4", temperature=0.1, max_tokens=512)

We will use [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) for embeddings.

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")

LLamaIndex core Settings update 

In [4]:
from llama_index.core import Settings
from llama_index.core.text_splitter import SentenceSplitter

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 2048

## Section-3: Load data
Downloaded from

https://www.infosys.com/investors/reports-filings/quarterly-results.html

In [5]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("./ifrs-inr-press-release.pdf")

# Define the output directory
output_dir = "output_documents"
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

# Define output file path
output_path = os.path.join(output_dir, "converted_document_infosys_pr_q1_2026.md")

# Save as Markdown
with open(output_path, "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

print(f"Document saved at: {output_path}")


[INFO] 2026-06-21 11:03:09,217 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-21 11:03:09,227 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-21 11:03:09,255 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-data-science-ai-nov-2025\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-21 11:03:09,256 [RapidOCR] main.py:50: Using C:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-data-science-ai-nov-2025\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-21 11:03:09,598 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-21 11:03:09,599 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-21 11:03:09,621 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-data-science-ai-nov-2025\.venv\Lib\site-packages\rapidocr\models

Document saved at: output_documents\converted_document_infosys_pr_q1_2026.md


In [6]:
from llama_index.core import SimpleDirectoryReader

q1_2026 = SimpleDirectoryReader(
    input_files=["./output_documents/converted_document_infosys_pr_q1_2026.md"]
).load_data()

## Section-4: Vector Storage Setup Using FAISS

In [7]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext

# 1. Define Embedding Dimension (Qwen3-Embedding-0.6B has 1024 dimensions)
d = 1024

# 2. Initialize a flat FAISS Index using L2 (Euclidean) distance
faiss_index = faiss.IndexFlatL2(d)

# 3. Wrap it in LlamaIndex's FaissVectorStore adapter
vector_store = FaissVectorStore(faiss_index=faiss_index)

# 4. Create a clean Storage Context layer
storage_context = StorageContext.from_defaults(vector_store=vector_store)

2026-06-21 11:04:56,615 - INFO - Loading faiss with AVX2 support.
2026-06-21 11:04:56,617 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-06-21 11:04:56,618 - INFO - Loading faiss.
2026-06-21 11:04:56,933 - INFO - Successfully loaded faiss.


##  Section-5: Building the RAG Pipeline

LlamaIndex simplifies the heavy lifting of segmenting documents into nodes, vectorizing them, and saving them into the vector store index.

In [8]:
from llama_index.core import VectorStoreIndex

# Create the index from our parsed Docling documents
# This pipeline splits texts, extracts vectors via Hugging Face, and stores them in FAISS.

q1_2026_index = VectorStoreIndex.from_documents(q1_2026, storage_context=storage_context)

## Section-6: Build query engines and Running Retrieval

In [9]:
# 1. Convert the index into a standard Query Engine
query_engine = q1_2026_index.as_query_engine(similarity_top_k=3)

In [10]:
# 2. Formulate your question

response = query_engine.query(
    "What is the basic EPS for the quarter ended June 30, 2025, and what was the YoY growth?"
)

2026-06-21 11:06:23,291 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:06:23,803 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:06:24,226 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [11]:
from llama_index.core.response.pprint_utils import pprint_response

pprint_response(response)

Final Response: The basic EPS for the quarter ended June 30, 2025, is
₹ 16.70, with a year-over-year (YoY) growth of 8.6%.


In [12]:
response = query_engine.query("Which are some of the key customer wins?")

2026-06-21 11:07:27,002 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:07:27,411 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [13]:
pprint_response(response)

Final Response: Some of the key customer wins include receiving the
Customer Innovation Award from Databricks for delivering impactful
solutions across industries, and being recognized as the Global System
Integrator of the Year-EMEA award at Stibo's PATH Summit 2025,
demonstrating the company's ability to drive innovation and deliver
value to its clients.


In [14]:
response = query_engine.query("What are some of the key achievements in the quarter?")

2026-06-21 11:07:58,537 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:07:58,962 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:07:59,570 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [15]:
pprint_response(response)

Final Response: Some of the key achievements in the quarter include
revenues growing by 3.8% year-over-year in constant currency terms and
2.6% quarter-over-quarter, with reported revenues reaching ₹ 42,279
crores, representing a 7.5% year-over-year growth. The operating
margin stood at 20.8%, and the basic EPS increased by 8.6% year-over-
year to ₹ 16.70. Additionally, the company achieved a free cash flow
of ₹ 7,533 crores, with a free cash flow conversion of 108.8% of net
profit. Large deal TCV also saw significant growth, with 55% being net
new, amounting to $3.8 billion.


In [16]:
response = query_engine.query("What is the total assets as of June 2025?")

2026-06-21 11:08:59,261 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:09:00,102 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:09:00,553 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:09:01,053 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:09:01,473 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [17]:
pprint_response(response)

Final Response: The total assets as of June 2025 is 193,517.


## Section-7: Inspecting the RAG Evidence

To ensure your RAG setup isn't hallucinating, you can verify exactly which document fragments the vector store pulled to formulate the answer (citation).

In [18]:
# Review source nodes pulled by FAISS
print("\n--- Source Documents Used ---")
for i, node in enumerate(response.source_nodes, 1):
    print(f"\n[Source Context Element {i}] (Score: {node.score:.4f}):")
    print(node.node.get_content()[:300] + "...")


--- Source Documents Used ---

[Source Context Element 1] (Score: 1.0513):
872 |
| Non-current investments                                    |          10,643 |           11,059 |
| Unbilled revenue                                           |           2,246 |            2,232 |
| Other non-current assets                                   |           6,952 |           ...

[Source Context Element 2] (Score: 1.1149):
These filings are available at www.sec.gov. Infosys may, from time to time, make additional written and oral forwardlooking  statements,  including  statements  contained  in  the  Company's  filings  with  the  Securities  and  Exchange Commission and our reports to shareholders. The Company does n...

[Source Context Element 3] (Score: 1.2707):
<!-- image -->

## Guidance for FY26:

- Revenue growth of 1%-3% in constant currency
- Operating margin of 20%-22%

## Key highlights:

## For the quarter ended June 30, 2025

- Revenues in CC terms grew by 3.8% YoY and by 2.6% 

## Section 8: Evaluation of RAG system using LLM as a Judge

The best approach is to replicate the two most critical metrics that evaluate the two completely separate halves of your RAG pipeline:

- **Faithfulness (Generation Layer):** Checks if the generated answer is strictly backed by the context (catches hallucinations).

- **Context Recall (Retrieval Layer):** Checks if your document retrieval actually found the right information compared to the ground truth.

In [19]:
from datasets import Dataset

# 1. Define evaluation questions and ground truths based on the data
eval_questions = [
    "What was the operating margin for Infosys in Q1, and how did it compare to the guidance?",
    "Who is the CFO of Infosys and what did he say about Project Maximus?",
    "Which award did Infosys BPM win at the Diversity Charter Awards 2025?"
]

ground_truths = [
    "The operating margin was 20.8%, which fell within the retained FY26 guidance range of 20%-22%.",
    "Jayesh Sanghrajka is the CFO. He stated that they continue to leverage Project Maximus to make investments in strategic priorities to drive profitable growth and enhance shareholder value.",
    "Infosys BPM won in the 'Employer Supporting Women in the Workplace' category for its HR initiative 'Empower with Care'."
]

# 2. Collect predictions from our LlamaIndex pipeline
answers = []
contexts = []

for query in eval_questions:
    # Query our existing LlamaIndex query engine
    response = query_engine.query(query)
    
    answers.append(response.response)
    # Extract the raw text from the retrieved source chunks
    contexts.append([node.node.get_content() for node in response.source_nodes])

# 3. Format the data into a dictionary for evaluation
data = {
    "question": eval_questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

2026-06-21 11:13:52,733 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:52,959 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:53,566 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:54,002 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:54,314 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:54,591 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:55,172 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-21 11:13:55,368 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [20]:
# Generated answers

answers

["The operating margin for the quarter was not explicitly provided in the given data. However, based on the information about profit before income taxes and other financial metrics, it's possible to analyze the company's profitability. The profit before income taxes was 9,740, and the net profit after non-controlling interest was 6,921. To determine the operating margin, additional data such as operating profit and gross profit would be necessary. Without this information, a direct comparison to the guidance cannot be made.",
 'The CFO of Infosys is Jayesh Sanghrajka. He said that they continue to leverage Project Maximus to make investments in strategic priorities to drive profitable growth and enhance shareholder value.',
 "Infosys BPM won the award in the 'Employer Supporting Women in the Workplace' category for its HR initiative, namely 'Empower with Care', at the Diversity Charter Awards 2025."]

In [21]:
import json
import re
from langchain_groq import ChatGroq

# 1. Initialize your LLM Judge
judge_llm = ChatGroq(
    model="openai/gpt-oss-20b", 
    temperature=0.0, 
    groq_api_key=os.environ["GROQ_API_KEY"]
)

# 2. Define the lightweight evaluation logic
def evaluate_rag_row(question, contexts, answer, ground_truth):
    # Combine the context list into a single continuous block of text
    context_str = "\n".join(contexts)
    
    judge_prompt = f"""
    You are an expert AI Software Quality Auditor assessing a Retrieval-Augmented Generation (RAG) system.
    Evaluate the provided RAG outputs across exactly two metrics.

    INPUT DATA:
    - QUESTION: {question}
    - RETRIEVED CONTEXT: {context_str}
    - GENERATED ANSWER: {answer}
    - GROUND TRUTH: {ground_truth}

    METRIC 1: FAITHFULNESS (Score 0.0 to 1.0)
    - Definition: Is every single factual claim made in the GENERATED ANSWER completely supported by and derived from the RETRIEVED CONTEXT?
    - If the answer includes numbers, names, or metrics NOT found in the context (even if true in the real world), it is a hallucination. Score it 0.0.
    - If it is entirely grounded, score it 1.0. Partial alignment can be scored between 0.0 and 1.0.

    METRIC 2: CONTEXT RECALL (Score 0.0 to 1.0)
    - Definition: Did the retrieval engine successfully fetch the information needed to form the absolute correct answer?
    - Compare the RETRIEVED CONTEXT against the GROUND TRUTH. If the core details present in the Ground Truth are missing from the retrieved text block, score it 0.0. 
    - If all required keys/metrics are perfectly retrieved, score it 1.0. Partially correct answers can be scored between 0.0 and 1.0.

    OUTPUT FORMAT:
    You must output your response exactly as a single, valid Python dictionary wrapper string. Do not include any intro, markdown block backticks, or outro text. 
    
    Format example:
    {{"faithfulness_score": 1.0, "faithfulness_reason": "Explanation text.", "context_recall_score": 0.0, "context_recall_reason": "Explanation text."}}
    """
    
    # Run the prompt through Groq
    response = judge_llm.invoke(judge_prompt).content
    
    # Strip away markdown block format backticks if the LLM accidentally adds them
    clean_response = re.sub(r"```(json|python)?|```", "", response).strip()
    
    try:
        return json.loads(clean_response)
    except Exception:
        # Fallback if JSON format has slight parsing discrepancies
        try:
            return eval(clean_response)
        except Exception:
            return {"error": "Failed to parse judge output", "raw": response}

In [22]:
# 3. Run Evaluation on own dataset

eval_results = []

print("Running Custom LLM-as-a-Judge Evaluation via Groq...")
for i in range(len(data["question"])):
    print(f"Evaluating Question {i+1}/{len(data['question'])}...")
    
    row_result = evaluate_rag_row(
        question=data["question"][i],
        contexts=data["contexts"][i],
        answer=data["answer"][i],
        ground_truth=data["ground_truth"][i]
    )
    
    # Append the metadata to create a comprehensive report
    row_result["question"] = data["question"][i]
    eval_results.append(row_result)

print("Evaluation complete!")



# 4. View results in a clean DataFrame

import pandas as pd
df = pd.DataFrame(eval_results)

# Reorder columns nicely for display
cols = ['question', 'faithfulness_score', 'faithfulness_reason', 'context_recall_score', 'context_recall_reason']
df[cols]

Running Custom LLM-as-a-Judge Evaluation via Groq...
Evaluating Question 1/3...


2026-06-21 11:17:42,263 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Evaluating Question 2/3...


2026-06-21 11:17:43,101 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Evaluating Question 3/3...


2026-06-21 11:17:43,895 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Evaluation complete!


,question,faithfulness_score,faithfulness_reason,context_recall_score,context_recall_reason
0,What was the operating margin for Infosys in Q...,0.0,The generated answer incorrectly claims that t...,1.0,The retrieved context includes all the key inf...
1,Who is the CFO of Infosys and what did he say ...,1.0,The generated answer only states the CFO's nam...,1.0,The retrieved context contains the CFO's name ...
2,Which award did Infosys BPM win at the Diversi...,1.0,The generated answer exactly matches the factu...,1.0,The retrieved context contains the award categ...


In [23]:
eval_results

[{'faithfulness_score': 0.0,
  'faithfulness_reason': 'The generated answer incorrectly claims that the operating margin was not explicitly provided in the retrieved context, yet the context contains the margin figure (20.8%). It also states that a direct comparison to the guidance cannot be made, even though the guidance (20%-22%) is present in the context. These unsupported or contradictory statements constitute hallucinations, so the answer is not faithful to the retrieved data.',
  'context_recall_score': 1.0,
  'context_recall_reason': 'The retrieved context includes all the key information needed for the correct answer: the operating margin of 20.8% and the FY26 guidance of 20%-22%. Therefore the retrieval engine successfully fetched the necessary details.',
  'question': 'What was the operating margin for Infosys in Q1, and how did it compare to the guidance?'},
 {'faithfulness_score': 1.0,
  'faithfulness_reason': "The generated answer only states the CFO's name and the exact q